# General AMD-stability criterion — example

Companion notebook for **amdrepertory**.

We compute, for a three-planet system:
- the optimal eccentricity-like vector $c = (c_1, c_2, c_3)$,
- the **actual** AMD,
- the **critical** AMD,
- the ratio $\beta = \mathrm{AMD}/\mathrm{AMD_{crit}}$  ($\beta<1$ stable, $\beta>1$ unstable).

Two paths are shown: (A) directly from periods/masses, and (B) from a `rebound` simulation.


In [1]:
# General modules import
import numpy as np

#Utilities
import os, sys

In [2]:
#Project specific modules
sys.path.append("amdrepertory_modules/")

import functions

from optimize import optimize_amd_crit, amd_crit_from_sim, AMDOptimizationError

## A. From periods and masses

Normalized units: central mass `m0=1`, `G=1`. Planet masses are then in units of the central mass.

In [3]:
periods = [1.0, 1.4, 1.8]      # P1, P2, P3
masses  = [1e-5, 1e-5, 1e-5]   # m1, m2, m3 (in units of m0)

out = optimize_amd_crit(periods, masses, verbose=True)

print()
print("c        =", out["c"])
print("AMD_crit =", out["AMD_crit"])
print("|tau-1|  =", out["tau_residual"])

IC#0: success=True c=[0.04621196 0.01632368 0.02458498] |tau-1|=1.27e-10 feasible=True msg='`gtol` termination condition is satisfied.'
IC#1: success=True c=[0.04627935 0.01631711 0.02455986] |tau-1|=1.29e-09 feasible=True msg='`gtol` termination condition is satisfied.'
IC#2: success=True c=[0.04611364 0.01637241 0.02456204] |tau-1|=2.02e-11 feasible=True msg='`gtol` termination condition is satisfied.'

c        = [0.04611364 0.01637241 0.02456204]
AMD_crit = 8.563001999660806e-09
|tau-1|  = 2.0215606966189625e-11


### Computing beta from an actual AMD

`optimize_amd_crit` gives `AMD_crit`. To get $\beta$ you need the system's
**actual** AMD. Below we compute it analytically from the eccentricities.

#### a. Stable

In [4]:
def amd_from_elements(periods, masses, eccs, m0=1.0, G=1.0):
    """Actual AMD = sum_i Lambda_i (1 - sqrt(1-e_i^2)), no inclination."""
    mu = G * m0
    amd = 0.0
    for P, m, e in zip(periods, masses, eccs):
        a = (mu * P**2 / (4*np.pi**2))**(1/3)
        Lam = m * (mu * a)**0.5
        amd += Lam * (1 - (1 - e**2)**0.5)
    return amd

eccs = [0.01, 0.005, 0.01]
AMD_actual = amd_from_elements(periods, masses, eccs)

beta = AMD_actual / out["AMD_crit"]
print(f"AMD (actual) = {AMD_actual:.4e}")
print(f"AMD_crit     = {out['AMD_crit']:.4e}")
print(f"beta         = {beta:.4f}  ->", "STABLE" if beta < 1 else "UNSTABLE")

AMD (actual) = 6.7637e-10
AMD_crit     = 8.5630e-09
beta         = 0.0790  -> STABLE


#### b. Unstable 

In [5]:
periods = [1.0, 1.5, 2.0]      # P1, P2, P3
masses  = [1e-4, 1e-5, 1e-5]   # m1, m2, m3 (in units of m0)
out = optimize_amd_crit(periods, masses, verbose=True)

eccs = [0.03, 0.04, 0.03]
AMD_actual = amd_from_elements(periods, masses, eccs)

beta = AMD_actual / out["AMD_crit"]
print(f"AMD (actual) = {AMD_actual:.4e}")
print(f"AMD_crit     = {out['AMD_crit']:.4e}")
print(f"beta         = {beta:.4f}  ->", "STABLE" if beta < 1 else "UNSTABLE")

IC#0: success=True c=[0.03289977 0.0156352  0.02826741] |tau-1|=2.02e-09 feasible=True msg='`gtol` termination condition is satisfied.'
IC#1: success=True c=[0.0299125  0.01687024 0.0284734 ] |tau-1|=3.06e-10 feasible=True msg='`gtol` termination condition is satisfied.'
IC#2: success=True c=[0.03261236 0.01568166 0.02842236] |tau-1|=1.88e-10 feasible=True msg='`gtol` termination condition is satisfied.'
AMD (actual) = 3.2430e-08
AMD_crit     = 3.2339e-08
beta         = 1.0028  -> UNSTABLE


## B. From a rebound simulation

`amd_crit_from_sim` reads masses and periods from the simulation, runs the
optimization, and returns everything including `beta` and `stable`.

> Requires `rebound`: `pip install rebound`

In [6]:
try:
    import rebound as rb

    sim = rb.Simulation()
    sim.G = 1.0
    sim.add(m=1.0)                                    # star
    sim.add(m=1e-5, P=1.0, e=0.01)                    # planet 1
    sim.add(m=1e-5, P=1.3, e=0.005)                   # planet 2
    sim.add(m=1e-5, P=1.7, e=0.01)                    # planet 3
    sim.move_to_com()

    res = amd_crit_from_sim(sim, with_inclination=False, verbose=False)
    print("c        =", res["c"])
    print("AMD       =", res["AMD"])
    print("AMD_crit  =", res["AMD_crit"])
    print("beta      =", res["beta"], "->", "STABLE" if res["stable"] else "UNSTABLE")

except ImportError:
    print("rebound not installed - skipping section B.")

c        = [0.02368572 0.01245542 0.02501051]
AMD       = 6.683022817375451e-10
AMD_crit  = 4.00180783507943e-09
beta      = 0.1670000932776624 -> STABLE


## C. Error handling

When the configuration cannot saturate `tau = 1` (e.g. very wide spacing or
large masses where the 3-body term alone exceeds 1), the optimizer raises a
clear `AMDOptimizationError` instead of returning a silently wrong result.

In [7]:
try:
    optimize_amd_crit(periods=[1.1, 1.4, 1.7], masses=[1e-5]*3)
except AMDOptimizationError as e:
    print("Caught expected error:")
    print(" ", e)

/opt/anaconda3/envs/stability/lib/python3.9/site-packages/scipy/optimize/_hessian_update_strategy.py:182: UserWarning: delta_grad == 0.0. Check if the approximated function is linear. If the function is linear better results can be obtained by defining the Hessian as zero instead of using quasi-Newton approximations.
  warn('delta_grad == 0.0. Check if the approximated '


Caught expected error:
  Optimization failed: no initial condition produced a feasible solution (tau=1 not satisfied within tolerance). Check the inputs or try method='trust-constr'.
